# Massive Activations in FLUX — Colab runner

Two experiments sharing one capture/ranking core:

- **Part 1 — Figure 3.** Top-/bottom-/random-k channel masks vs. a BiRefNet pseudo-GT, and the layer-wise mIoU curve (Fig 3D). Model is swappable — pick a preset in §5.
- **Part 2 — Channel stability.** For each (prompt, seed), at a fixed block + timestep, how much do the top channel *identities* change across generations? See `src/experiments/channel_stability.py`.

Sections 1–4 (setup, install, HF auth) are shared. Run Part 1 (§5–8) and/or Part 2 (§9–11) independently; §12 downloads results.

**Before you run:** `Runtime > Change runtime type` → GPU, and add an `HF_TOKEN` under Colab's 🔑 *Secrets* (needed for gated checkpoints like FLUX.1-dev — accept its license on the model page first).


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader \
    || echo "No GPU detected — set Runtime > Change runtime type > GPU"
import sys

print(sys.version)

## 1. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/BrendanGho/massive-activations-fig3.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
REPO_DIR = "/content/massive-activations-fig3"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists — pulling latest {BRANCH}...")
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}

## 2. (Optional) Mount Google Drive

Keeps outputs across sessions. Everything below is written under `DRIVE_ROOT/<model>/...` — each model gets its own cache + output tree (see §5).


In [ ]:
USE_DRIVE = True  # @param {type:"boolean"}

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/Research/MA"
else:
    DRIVE_ROOT = "/content/figure3_repro"

import os

os.makedirs(DRIVE_ROOT, exist_ok=True)
print("DRIVE_ROOT:", DRIVE_ROOT)


## 3. Install dependencies

Editable install with the `fig3` extra (torch / diffusers / transformers / scikit-learn / matplotlib / ...). Covers both parts.


In [ ]:
!pip install -q -e ".[fig3]"

In [ ]:
import torch

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    major, _minor = torch.cuda.get_device_capability(0)
    DEVICE = "cuda"
    # bf16 needs Ampere+ (compute capability >= 8, e.g. A100/L4); T4 (cc 7.5) falls back to fp16.
    DTYPE = "bf16" if major >= 8 else "fp16"
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
        "| compute capability:",
        major,
        "| using dtype:",
        DTYPE,
    )
else:
    DEVICE = "cpu"
    DTYPE = "fp32"
    print("No GPU — falling back to CPU/fp32. This will be extremely slow; use a GPU runtime.")

## 4. Hugging Face authentication

Add `HF_TOKEN` under Colab's 🔑 *Secrets* (persists across sessions), or log in interactively below.


In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token)
    print("Logged in via Colab secret HF_TOKEN.")
else:
    login()  # interactive prompt

# Part 1 — Figure 3

Reproduce the qualitative masks (Fig 3A–C) and the layer-wise mIoU curve (Fig 3D).


## 5. Pick a model and configure the run

`MODEL_PRESETS` holds the per-model knobs that matter for generation quality (denoising
steps, guidance scale, offload threshold) — change `ACTIVE_MODEL` to switch models, no
need to touch anything else. Add a new key to try another checkpoint.

Everything Part 1 writes goes under `DRIVE_ROOT/<model>/<smoke|full>/{cache,outputs}` —
each model and each run size (smoke vs. full) gets its own isolated cache/output dir, so
a full run can never mistake stale smoke-test cache entries for its own (the activation
cache is keyed by prompt *position*, so sharing a dir across runs would mislabel prompts).


In [ ]:
import yaml

# Per-model generation settings. flux-schnell/flux1-dev are NOT distilled the same way as
# klein: dev needs CFG + more steps to converge, schnell is guidance-distilled for few steps.
# Keeping these here (not hard-coded in configs/*.yaml) is what makes swapping models a
# one-line change.
MODEL_PRESETS = {
    "flux2-klein": {
        "model_ckpt": "black-forest-labs/FLUX.2-klein-4B",
        "num_denoising_steps": 4,
        "guidance_scale": None,
        "offload_below_gb": 24,  # ~16GB in half precision
    },
    "flux-schnell": {
        "model_ckpt": "black-forest-labs/FLUX.1-schnell",
        "num_denoising_steps": 4,
        "guidance_scale": None,
        "offload_below_gb": 24,
    },
    "flux1-dev": {
        "model_ckpt": "black-forest-labs/FLUX.1-dev",  # gated — accept the license first
        "num_denoising_steps": 25,
        "guidance_scale": 3.5,
        "offload_below_gb": 40,  # ~24GB in half precision
    },
}

ACTIVE_MODEL = "flux2-klein"  # @param ["flux2-klein", "flux-schnell", "flux1-dev"]
BIREFNET_WEIGHTS = "ZhengPeng7/BiRefNet"  # @param {type:"string"}

preset = MODEL_PRESETS[ACTIVE_MODEL]
MODEL_DIR = f"{DRIVE_ROOT}/{ACTIVE_MODEL}"
SMOKE_OUTPUT_DIR = f"{MODEL_DIR}/smoke/outputs"
SMOKE_CACHE_DIR = f"{MODEL_DIR}/smoke/cache"
os.makedirs(SMOKE_OUTPUT_DIR, exist_ok=True)
os.makedirs(SMOKE_CACHE_DIR, exist_ok=True)

# 1,600 GenAI-Bench prompts ship at data/genai_prompts.jsonl for the full run (§7);
# the smoke test always uses a small built-in prompt list.
PROMPT_SOURCE = f"{SMOKE_OUTPUT_DIR}/smoke_test_prompts.txt"
_SMOKE_PROMPTS = [
    "a red bicycle leaning against a brick wall",
    "a golden retriever sitting on a wooden dock at sunset",
    "a bowl of fresh strawberries on a marble countertop",
    "an astronaut planting a flag on the moon",
]
if not os.path.isfile(PROMPT_SOURCE):
    with open(PROMPT_SOURCE, "w") as fh:
        fh.write("\n".join(_SMOKE_PROMPTS) + "\n")

if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    offload = total_gb < preset["offload_below_gb"]
    print(f"GPU memory: {total_gb:.1f} GB -> cpu offload: {offload}")
else:
    offload = False

config = {
    "model_ckpt": preset["model_ckpt"],
    "prompt_source": PROMPT_SOURCE,
    "birefnet_weights": BIREFNET_WEIGHTS,
    "output_dir": SMOKE_OUTPUT_DIR,
    "activation_cache_dir": SMOKE_CACHE_DIR,
    "num_denoising_steps": preset["num_denoising_steps"],
    "guidance_scale": preset["guidance_scale"],
    "resolution": 1024,
    "top_k": 12,
    "layers": "all",
    "random_k_trials": 5,
    "seed": 0,
    "device": DEVICE,
    "dtype": DTYPE,
    "offload": offload,
    "batch_size": 1,
    "num_example_prompts": 8,
    "cache_batch_size": 32,
}

CONFIG_PATH = "configs/colab.yaml"
with open(CONFIG_PATH, "w") as fh:
    yaml.safe_dump(config, fh, sort_keys=False)
print(open(CONFIG_PATH).read())


## 6. Quick smoke test

Runs 4 built-in prompts end to end — catches config, auth, or model-loading issues in minutes.


In [ ]:
!python -m src.stage1_generate_and_cache --config {CONFIG_PATH} --fused --limit 4
!python -m src.stage4_evaluate_figure3d --config {CONFIG_PATH}

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f"{SMOKE_OUTPUT_DIR}/figure3d_results.csv")
display(df.pivot(index="layer", columns="strategy", values="mean_miou"))
display(Image(filename=f"{SMOKE_OUTPUT_DIR}/figure3d_curve.png"))


## 7. Full run (1,600 GenAI-Bench prompts)

Uses the bundled `data/genai_prompts.jsonl` — no download needed. Writes to
`DRIVE_ROOT/<model>/full/...`, separate from the smoke-test dir from §5, so it's always a
clean cache regardless of what smoke tests you've run for this model. Resumable: re-running
this cell skips prompts already in `full/cache`'s `completed.json`.


In [ ]:
FULL_OUTPUT_DIR = f"{MODEL_DIR}/full/outputs"
FULL_CACHE_DIR = f"{MODEL_DIR}/full/cache"
os.makedirs(FULL_OUTPUT_DIR, exist_ok=True)
os.makedirs(FULL_CACHE_DIR, exist_ok=True)

full_config = dict(config)  # reuse the model/dtype/offload knobs from section 5
full_config.update(
    prompt_source="data/genai_prompts.jsonl",
    output_dir=FULL_OUTPUT_DIR,
    activation_cache_dir=FULL_CACHE_DIR,
)
FULL_CONFIG_PATH = "configs/colab_full.yaml"
with open(FULL_CONFIG_PATH, "w") as fh:
    yaml.safe_dump(full_config, fh, sort_keys=False)
print(open(FULL_CONFIG_PATH).read())

!bash scripts/run_pipeline.sh {FULL_CONFIG_PATH}


## 8. Results

In [ ]:
import glob

import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f"{FULL_OUTPUT_DIR}/figure3d_results.csv")
display(df.pivot(index="layer", columns="strategy", values="mean_miou"))
display(Image(filename=f"{FULL_OUTPUT_DIR}/figure3d_curve.png"))

for qdir in sorted(glob.glob(f"{FULL_OUTPUT_DIR}/qualitative/prompt_*"))[:3]:
    print(qdir)
    for png in sorted(glob.glob(f"{qdir}/*.png"))[:4]:
        display(Image(filename=png, width=220))


# Part 2 — Per-generation channel stability

*"Which channels are the largest for a given generation, and do they change depending on
what's being generated / from one generation to the next?"*

Fixes **one block (layer 11)** and the **last denoising step**, runs a `prompts x seeds`
scenario grid, ranks channels per scenario by `abs(mean_over_tokens)`, and checks whether
top-channel identities are stable. Requires §1–4 above. Defaults to FLUX.1-dev — **gated +
12B**, accept its license and prefer an A100/L4 (auto-offloads on smaller GPUs).


## 9. Configure the channel-stability run

In [ ]:
import yaml

CS_MODEL_CKPT = "black-forest-labs/FLUX.1-dev"  # @param {type:"string"}
CS_FIXED_LAYER = 11  # @param {type:"integer"}
CS_MODEL_SLUG = "flux1-dev"  # used only for the Drive folder name below
CS_OUTPUT_DIR = f"{DRIVE_ROOT}/{CS_MODEL_SLUG}/channel_stability"
os.makedirs(CS_OUTPUT_DIR, exist_ok=True)

# 12B in bf16 ~= 24 GB. Offload unless we detect a large GPU (A100-40GB+).
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    CS_OFFLOAD = total_gb < 40.0
    print(f"GPU memory: {total_gb:.1f} GB -> cpu offload: {CS_OFFLOAD}")
else:
    CS_OFFLOAD = True

cs_config = {
    "model_ckpt": CS_MODEL_CKPT,
    "output_dir": CS_OUTPUT_DIR,
    "resolution": 1024,
    "num_denoising_steps": 50,
    "guidance_scale": 3.5,
    "device": DEVICE,
    "dtype": DTYPE,
    "offload": CS_OFFLOAD,
    "fixed_layer": CS_FIXED_LAYER,
    # scenarios = prompts x seeds. Same prompt/diff seed => generation-to-generation
    # jitter; different prompt => content dependence of the top channels.
    "prompts": [
        "a red bicycle leaning against a brick wall",
        "a golden retriever sitting on a wooden dock at sunset",
        "a bowl of fresh strawberries on a marble countertop",
        "an astronaut planting a flag on the moon",
    ],
    "seeds": [0, 1, 2],
    "top_n": 20,
    "agg_k": 10,
    "n_channel_maps": 5,
    "n_control_maps": 0,  # set >0 to also dump low-ranked control channel maps
}

CS_CONFIG_PATH = "configs/channel_stability_colab.yaml"
with open(CS_CONFIG_PATH, "w") as fh:
    yaml.safe_dump(cs_config, fh, sort_keys=False)

print(open(CS_CONFIG_PATH).read())
print(
    f"scenarios: {len(cs_config['prompts'])} prompts x {len(cs_config['seeds'])} seeds "
    f"= {len(cs_config['prompts']) * len(cs_config['seeds'])}"
)


## 10. Run the experiment\n\n12 scenarios x 50 steps on the 12B model — budget GPU time (much longer with CPU offload).

In [ ]:
!python -m src.experiments.channel_stability --config {CS_CONFIG_PATH}

## 11. Channel-stability results

The `stability_summary.json` line is the answer: if top-channel identities are prompt-driven, same-prompt/different-seed Jaccard should exceed different-prompt Jaccard.


In [ ]:
import glob
import json

import pandas as pd
from IPython.display import Image, display

summary = json.load(open(f"{CS_OUTPUT_DIR}/stability_summary.json"))
print(f"scenarios: {summary['n_scenarios']} | seeds: {summary['n_seeds']}")
for k, v in summary["per_k"].items():
    print(
        f"top-{k:>2}: {v['n_distinct_channels_used']:>3} distinct channels used | "
        f"Jaccard same-prompt/diff-seed={v['mean_jaccard_same_prompt_diff_seed']} "
        f"diff-prompt={v['mean_jaccard_diff_prompt']}"
    )

print("\nfirst rows of the ordered top-20 table:")
display(pd.read_csv(f"{CS_OUTPUT_DIR}/channel_stability_topk.csv").head(20))

overlap = f"{CS_OUTPUT_DIR}/stability_overlap.png"
if os.path.isfile(overlap):
    display(Image(filename=overlap))

for sdir in sorted(glob.glob(f"{CS_OUTPUT_DIR}/scenarios/scenario_*")):
    print(sdir)
    display(Image(filename=f"{sdir}/image.png", width=256))
    pngs = sorted(glob.glob(f"{sdir}/channel_rank*.png")) + sorted(
        glob.glob(f"{sdir}/aggregated_*.png")
    )
    for png in pngs:
        display(Image(filename=png, width=200))

## 12. (Optional) Download outputs

Only needed if you skipped Drive in §2 — otherwise outputs already live under `DRIVE_ROOT`.


In [ ]:
if not USE_DRIVE:
    import shutil

    from google.colab import files

    dirs = [("figure3_outputs", OUTPUT_DIR)]
    if "CS_OUTPUT_DIR" in globals():
        dirs.append(("channel_stability", CS_OUTPUT_DIR))
    for name, d in dirs:
        if os.path.isdir(d) and os.listdir(d):
            archive = shutil.make_archive(f"/content/{name}", "zip", d)
            files.download(archive)
else:
    print("Outputs are already in Google Drive under:", DRIVE_ROOT)